# SENTINEL — Train hành vi "đánh nhau" (luồng B: khung xương + ST-GCN)

Pipeline: tải dataset → loại video trùng → trích khung xương YOLOv8n-pose ở imgsz 1280 + BoT-SORT (10 FPS) → train ST-GCN theo cặp người (MIL) → đánh giá trên tập test.

- **Bộ chính:** UBI-Fights (1.000 video, nhãn theo frame, có tập test chuẩn `test_videos.csv`).
- **Train thêm:** Real Life Violence Situations (Kaggle, 2.000 clip) — bật mặc định, cần `kaggle.json`.
- **Test chéo:** Surveillance Camera Fight (300 clip, MIT).

**Lần 3 (`fight_v3`) — thay đổi so với lần 1–2** (lần 2: AUC theo frame 0,61; trong từng video đánh nhau model không phân biệt đoạn đánh với đoạn yên, AUC trung vị 0,52; điểm tăng theo số người trong cảnh):

| Quyết định cũ | Lần 3 | Lý do |
|---|---|---|
| Train: video đánh nhau chỉ lấy đoạn đánh ±10 s; video bình thường 20 s đầu (dùng ~7,7 / 66 giờ) | Trích **toàn bộ** video | Phần bình thường trong video đánh nhau: 41% → 100%; video bình thường: 7% → 100%. Mẫu âm đa dạng hơn, bớt overfit (lần 2: val loss tăng từ epoch 4) |
| Test: video bình thường 60 s đầu | Toàn bộ tập test (`official_protocol = true`); thêm nhóm `_short` giống tập test cũ để so với lần 2 | Số đo tin cậy hơn |
| Validation trên cửa sổ đã cắt, chọn checkpoint/ngưỡng theo AUC cửa sổ | Validation trên **video đầy đủ**, AUC theo frame (giống cách tính ở tập test) | Checkpoint và ngưỡng chọn đúng thước đo của test |
| Lấy mẫu 50% đánh nhau / 50% bình thường | 50% đánh nhau, 25% bình thường trong video đánh nhau, 25% video bình thường; từ epoch 3, một nửa phần bình thường ưu tiên đoạn bị chấm điểm cao (hard-negative mining) | Model phải học phân biệt trong cùng khung cảnh, không chỉ phân biệt video |
| Cửa sổ bình thường sát ranh giới đánh nhau | Bỏ cửa sổ bình thường cách đoạn đánh nhau < 1 s | Nhãn ranh giới không chính xác |
| Điểm cửa sổ = max điểm các cặp | Smooth max chuẩn hóa theo số cặp (`POOL_R`) | Max tăng theo số cặp → cảnh đông bị chấm như đánh nhau |
| Chỉ UBI-Fights (`USE_RLVS = False`) | Thêm RLVS | 176 video đánh nhau là quá ít |

**Giữ nguyên vì phải khớp với máy chạy thật (CPU local):** YOLOv8n-pose, imgsz 1280, 10 FPS, cửa sổ 2,4 s, tối đa 16 cặp/cửa sổ, bán kính 1,5 lần chiều cao người. Đổi các thông số này thì model train trên Colab sẽ không khớp với khung xương mà dashboard trích được.

**Khối lượng:** ~80 giờ video cần trích khung xương (lần 1: 9,7 giờ trong 47 phút), dự kiến 6–8 giờ trên L4; train tối đa 40 epoch khoảng 2–3 giờ (dừng sớm nếu validation không cải thiện sau 10 epoch). Mọi bước chạy tiếp được sau khi Colab ngắt: khung xương lưu từng video lên Drive, train lưu checkpoint mỗi epoch. Chỉ lưu khung xương và model lên Drive, không lưu video đã giải nén.

**GPU:** `Runtime > Change runtime type > L4 GPU`. Trích khung xương chạy song song nhiều tiến trình trên cùng GPU (số tiến trình theo số CPU của máy Colab).

**Treo máy:** giữ `AUTO_DISCONNECT = True` (mặc định) rồi `Runtime > Run all`. Chạy xong — hoặc gặp lỗi ở bất kỳ mục nào — notebook ghi lý do vào `MyDrive/SENTINEL/artifacts/<RUN_NAME>/run_log.txt`, đồng bộ Drive rồi tự ngắt runtime để không tốn compute unit. Giữ tab trình duyệt mở: Colab Pro không chạy nền khi đóng tab.

## 0. Cấu hình

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
BRANCH = "main"
DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_DIR = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
ZIP_DIR = DRIVE_ROOT / "datasets" / "fight"          # chỉ giữ file zip gốc
POSE_ROOT = DRIVE_ROOT / "features" / "pose_v1"       # khung xương đã trích (dùng lại cache của lần 1–2)
RUN_NAME = "fight_v3"          # mỗi lần train lại đặt tên mới để giữ kết quả cũ mà so sánh
PREVIOUS_RUN = "fight_v2"      # lần chạy trước (để so sánh ở mục 11); "" nếu không có
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / RUN_NAME
DEDUP_DIR = DRIVE_ROOT / "artifacts" / "fight" / "dedup"   # dùng chung giữa các lần chạy (giữ cache hash)
LOCAL_DATA = Path("/content/datasets")                 # video giải nén (mất khi hết phiên)

UBI_URL = "https://socia-lab.di.ubi.pt/EventDetection/UBI_FIGHTS.zip"
UBI_ZIP_BYTES = 8_117_112_419
SCF_URL = "https://github.com/sayibet/fight-detection-surv-dataset.git"
KEEP_UBI_ZIP_ON_DRIVE = True   # False: tải lại mỗi phiên, tiết kiệm 8,1 GB Drive
USE_RLVS = True                # cần MyDrive/SENTINEL/kaggle.json (mục 4b); thiếu file thì bỏ qua RLVS và ghi vào run_log

# Phần video được trích khung xương; None = toàn bộ video.
# Lần 1–2 dùng 10 / 20 / 60 để tiết kiệm (~9,7 thay vì ~79 giờ video) và làm model kém đi.
FIGHT_MARGIN_SECONDS = None    # train, video đánh nhau: chỉ giữ đoạn đánh nhau ± N giây
MAX_SECONDS_NORMAL = None      # train, video bình thường: N giây đầu
TEST_NORMAL_SECONDS = None     # test, video bình thường: N giây đầu
SHORT_TEST_SECONDS = 60        # báo cáo thêm nhóm "_short" (video test bình thường cắt 60 s như lần 1–2) để so sánh
EVAL_STRIDE = 3                # chấm 1 cửa sổ mỗi 0,3 giây khi test; frame ở giữa được nội suy

# Train (xem mục 8)
EPOCHS = 40
SAMPLES_PER_EPOCH = 32_768     # cửa sổ mỗi epoch, một nửa là đánh nhau (~1 lượt qua các cửa sổ đánh nhau); lần 1–2: 16.384
POOL_R = 5.0                   # gộp điểm các cặp: smooth max chuẩn hóa theo số cặp; 0 = max như lần 1–2
HARD_NEGATIVE_FRACTION = 0.5   # tỉ lệ mẫu bình thường lấy theo điểm model (từ epoch 3)
VAL_STRIDE = 5                 # validation: chấm 1 cửa sổ mỗi 0,5 giây trên video đầy đủ

AUTO_DISCONNECT = True         # treo máy: chạy xong hoặc gặp lỗi thì ghi log, đồng bộ Drive rồi ngắt runtime

UBI_ROOT = LOCAL_DATA / "UBI_FIGHTS"
SCF_ROOT = LOCAL_DATA / "fight-detection-surv-dataset"
RLVS_ROOT = LOCAL_DATA / "rlvs"
print("Pose cache:", POSE_ROOT)
print("Artifacts:", ARTIFACT_DIR)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for path in [REPO_DIR.parent, ZIP_DIR, POSE_ROOT, ARTIFACT_DIR, LOCAL_DATA]:
    path.mkdir(parents=True, exist_ok=True)

import time
from contextlib import contextmanager
RUN_LOG = ARTIFACT_DIR / "run_log.txt"

def note(message):
    # Ghi một dòng vào log trên Drive (không ngắt runtime).
    with open(RUN_LOG, "a", encoding="utf-8") as log:
        log.write(f"{time.strftime('%Y-%m-%d %H:%M:%S')} {message}\n")
    print(message)

def finish_and_disconnect(message):
    # Ghi log lên Drive; khi AUTO_DISCONNECT thì đồng bộ Drive rồi ngắt runtime để dừng tốn compute unit.
    note(message)
    if AUTO_DISCONNECT:
        from google.colab import runtime
        drive.flush_and_unmount()
        runtime.unassign()

@contextmanager
def step(name):
    try:
        yield
    except Exception as error:
        finish_and_disconnect(f"LỖI ở mục {name}: {error!r} — xem output của mục này, sửa rồi chạy lại.")
        raise

print("Log chạy:", RUN_LOG, "| AUTO_DISCONNECT =", AUTO_DISCONNECT)

## 2. Clone lần đầu / đồng bộ code mới nhất
GitHub là nguồn chuẩn: cell bỏ mọi thay đổi trong file tracked trên Drive rồi đồng bộ với `origin`.

In [ ]:
with step('2. Đồng bộ code'):
    import os, subprocess

    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["git", "log", "--oneline", "-1"], check=True)

## 3. Cài dependency và kiểm tra GPU

In [ ]:
with step('3. Cài dependency'):
    import sys, torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    CPUS = os.cpu_count() or 2
    POSE_SHARDS = max(1, min(6, CPUS - 1))   # tiến trình trích khung xương chạy song song
    NUM_WORKERS = max(1, CPUS - 1)           # tiến trình dựng cửa sổ khi train
    print("PyTorch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "KHÔNG CÓ GPU")
    print(f"CPU: {CPUS} | pose shards: {POSE_SHARDS} | train workers: {NUM_WORKERS}")
    (REPO_DIR / ".env").write_text(f"YOLO_DEVICE={DEVICE}\n", encoding="utf-8")

## 4. Tải dataset

**UBI-Fights:** chứng chỉ SSL của trang tác giả đã hết hạn nên dùng `--no-check-certificate`. File zip 8,1 GB; `-c` cho phép tải tiếp khi bị ngắt.

In [ ]:
with step('4. Tải dataset'):
    ubi_zip = (ZIP_DIR if KEEP_UBI_ZIP_ON_DRIVE else Path("/content")) / "UBI_FIGHTS.zip"
    if not ubi_zip.exists() or ubi_zip.stat().st_size != UBI_ZIP_BYTES:
        subprocess.run(["wget", "-c", "--no-check-certificate", "-q", "--show-progress", UBI_URL, "-O", str(ubi_zip)], check=True)
    assert ubi_zip.stat().st_size == UBI_ZIP_BYTES, "File zip UBI-Fights chưa tải đủ"
    if not (UBI_ROOT / "test_videos.csv").exists():
        subprocess.run(["unzip", "-q", "-o", str(ubi_zip), "-d", str(LOCAL_DATA)], check=True)
    print("UBI-Fights videos:", len(list(UBI_ROOT.glob("videos/*/*.mp4"))))

    if not SCF_ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", SCF_URL, str(SCF_ROOT)], check=True)
    print("Surveillance Camera Fight videos:", len(list(SCF_ROOT.rglob("*.mp4"))) + len(list(SCF_ROOT.rglob("*.avi"))))

### 4b. Real Life Violence Situations
Tải `kaggle.json` từ trang tài khoản Kaggle, upload vào `MyDrive/SENTINEL/kaggle.json`. Thiếu file thì notebook train không có RLVS và ghi cảnh báo vào `run_log.txt` thay vì dừng. Dataset chưa ghi giấy phép rõ ràng, tác giả yêu cầu trích dẫn; chỉ dùng cho nghiên cứu.

In [ ]:
with step('4b. RLVS'):
    import shutil
    DATASETS_TRAIN = [f"ubi:{UBI_ROOT}"]
    kaggle_json = DRIVE_ROOT / "kaggle.json"
    if USE_RLVS and not kaggle_json.exists():
        note(f"CẢNH BÁO: không có {kaggle_json} — lần train này KHÔNG dùng RLVS.")
    elif USE_RLVS:
        kaggle_dir = Path.home() / ".kaggle"; kaggle_dir.mkdir(exist_ok=True)
        shutil.copy2(kaggle_json, kaggle_dir / "kaggle.json")
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
        if shutil.which("kaggle") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        if not RLVS_ROOT.exists():
            subprocess.run(["kaggle", "datasets", "download", "-d", "mohamedmustafa/real-life-violence-situations-dataset",
                            "-p", str(RLVS_ROOT), "--unzip"], check=True)
        DATASETS_TRAIN.append(f"rlvs:{RLVS_ROOT}")
    DATASETS_TEST = [f"ubi:{UBI_ROOT}", f"scf:{SCF_ROOT}"]
    DATASETS_ALL = DATASETS_TRAIN + [f"scf:{SCF_ROOT}"]
    print(DATASETS_ALL)

## 5. Smoke test
Trích khung xương 3 video test để chắc chắn YOLOv8-pose nhận ra người trước khi chạy toàn bộ.

In [ ]:
with step('5. Smoke test'):
    def run(script, *args):
        command = [sys.executable, "-u", script, *[str(a) for a in args]]
        subprocess.run(command, cwd=REPO_DIR, check=True)

    def dataset_args(specs):
        return [part for spec in specs for part in ("--dataset", spec)]

    smoke_root = Path("/content/pose_smoke")
    run("scripts/extract_pose.py", *dataset_args([f"ubi:{UBI_ROOT}"]), "--pose-root", smoke_root,
        "--split", "test", "--limit", "3", "--device", DEVICE, "--force")
    import json
    videos = json.loads((smoke_root / "ubi" / "manifest.json").read_text())["videos"]
    for key, info in videos.items():
        print(f"{key}: {info['samples']} mẫu, {info['person_rows'] / max(info['samples'], 1):.1f} người/mẫu, {info['tracks']} track")
    assert all(info["person_rows"] > 0 for info in videos.values()), "Không phát hiện được người — kiểm tra lại model/imgsz"

## 6. Loại video trùng giữa các dataset
Chỉ cặp khác dataset bị loại khỏi train; cặp trong cùng dataset chỉ ghi vào báo cáo để giữ tập chia chuẩn của tác giả.

In [ ]:
with step('6. Loại video trùng'):
    run("scripts/dedup_videos.py", *dataset_args(DATASETS_ALL), "--output-dir", DEDUP_DIR)
    report = json.loads((DEDUP_DIR / "exclude.json").read_text())
    print("Cặp trùng khác dataset:", len(report["duplicate_pairs"]))
    print("Loại khỏi train:", report["exclude_from_training"][:20])

## 7. Trích khung xương (song song)
`POSE_SHARDS` tiến trình cùng dùng GPU (FP16), mỗi tiến trình nhận một phần danh sách video và ghi log riêng ở `/content/pose_shard*.log`. Lần 3 trích toàn bộ video: cache của lần 1–2 đã đầy đủ (video test đánh nhau, Surveillance Camera Fight) được dùng lại; video đã bị cắt được trích lại từ đầu. Khoảng 80 giờ video, dự kiến 6–8 giờ. Video gốc không bị sửa. Chạy lại cell này sau khi Colab ngắt: video đã có cache đủ được bỏ qua.

In [ ]:
with step('7. Trích khung xương'):
    import time
    extract_options = []
    for flag, value in (("--fight-margin-seconds", FIGHT_MARGIN_SECONDS), ("--max-seconds-normal", MAX_SECONDS_NORMAL),
                        ("--max-seconds-test-normal", TEST_NORMAL_SECONDS)):
        if value is not None:
            extract_options += [flag, str(value)]
    print("Tùy chọn cắt video:", extract_options or "không cắt (toàn bộ video)")
    processes = []
    for shard in range(POSE_SHARDS):
        log = open(f"/content/pose_shard{shard}.log", "w")
        command = [sys.executable, "-u", "scripts/extract_pose.py", *dataset_args(DATASETS_ALL),
                   "--pose-root", str(POSE_ROOT), "--split", "all", "--device", DEVICE,
                   *extract_options, "--shard", f"{shard}/{POSE_SHARDS}"]
        processes.append((subprocess.Popen(command, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT), log))

    def last_line(shard):
        lines = Path(f"/content/pose_shard{shard}.log").read_text(errors="ignore").strip().splitlines()
        return lines[-1] if lines else "(chưa có output)"

    started = time.time()
    while any(process.poll() is None for process, _ in processes):
        time.sleep(300)
        print(f"[{(time.time() - started) / 60:.0f} phút]")
        for shard in range(POSE_SHARDS):
            print(f"   shard {shard}: {last_line(shard)[:110]}")
    for shard, (process, log) in enumerate(processes):
        log.close()
        if process.returncode != 0:
            print(Path(f"/content/pose_shard{shard}.log").read_text(errors="ignore")[-3000:])
            raise RuntimeError(f"Shard {shard} lỗi (mã {process.returncode}); chạy lại cell để tiếp tục")
    print("Trích khung xương xong:", sum(1 for _ in POSE_ROOT.glob("*/*.npz")), "video")

## 7b. Chẩn đoán dữ liệu train
Chạy trên khung xương đã có (không trích lại):
1. **Độ phủ:** tỉ lệ cửa sổ đánh nhau / bình thường có ít nhất một cặp người, số cặp, độ tin cậy khớp xương.
2. **Đường cơ sở thủ công:** AUC của điểm tương tác (tay chân chuyển động mạnh, sát người kia) dùng thẳng làm điểm đánh nhau. Lần 2: 0,59.
3. **Thử overfit:** train model mới, không tăng cường dữ liệu, trên 256 cửa sổ; phải học thuộc (AUC gần 1) — nếu không là có lỗi dữ liệu/code.

Kết quả lưu ở `ARTIFACT_DIR/diagnostics/diagnostics.json`; mục này không chặn các mục sau.

In [ ]:
with step('7b. Chẩn đoán'):
    run("scripts/diagnose_fight.py", *dataset_args(DATASETS_TRAIN), "--pose-root", POSE_ROOT,
        "--output-dir", ARTIFACT_DIR / "diagnostics", "--exclude", DEDUP_DIR / "exclude.json", "--device", DEVICE)

## 8. Train ST-GCN
- **Mẫu train:** toàn bộ video; cửa sổ bình thường cách đoạn đánh nhau dưới 1 s bị bỏ (nhãn ranh giới không chính xác).
- **Lấy mẫu mỗi epoch:** 50% cửa sổ đánh nhau, 25% cửa sổ bình thường trong video có đánh nhau, 25% video bình thường. Từ epoch 3, `HARD_NEGATIVE_FRACTION` của mỗi nhóm bình thường được lấy theo điểm model chấm gần nhất (đoạn bình thường bị chấm cao được học lại nhiều hơn).
- **Điểm cửa sổ:** smooth max chuẩn hóa theo số cặp (`POOL_R`), không tăng khi thêm cặp người bình thường.
- **Validation:** 15% video train, chấm trên video đầy đủ và tính AUC theo frame như tập test; chọn checkpoint theo chỉ số này, ngưỡng theo balanced accuracy trên frame. Log mỗi epoch có thêm AUC trung vị trong từng video đánh nhau (lần 2: 0,52 trên tập test).

Luôn chạy với `--resume`: có checkpoint thì train tiếp, chưa có thì bắt đầu mới; lần train đã xong không bị train thêm khi chạy lại. Mỗi lần muốn train lại từ đầu, đổi `RUN_NAME` ở mục 0.

In [ ]:
with step('8. Train'):
    run("scripts/train_fight.py", *dataset_args(DATASETS_TRAIN), "--pose-root", POSE_ROOT,
        "--output-dir", ARTIFACT_DIR, "--exclude", DEDUP_DIR / "exclude.json",
        "--epochs", EPOCHS, "--samples-per-epoch", SAMPLES_PER_EPOCH, "--batch-size", 64,
        "--pool-r", POOL_R, "--hard-negative-fraction", HARD_NEGATIVE_FRACTION, "--val-stride", VAL_STRIDE,
        "--num-workers", NUM_WORKERS, "--device", DEVICE, "--resume")

## 9. Báo cáo training

In [ ]:
with step('9. Báo cáo training'):
    import matplotlib.pyplot as plt
    config = json.loads((ARTIFACT_DIR / "config.json").read_text())
    threshold = json.loads((ARTIFACT_DIR / "fight_threshold.json").read_text())
    history = json.loads((ARTIFACT_DIR / "training_history.json").read_text())["history"]
    for key in ["train_windows", "train_fight_windows", "train_normal_windows_in_fight_videos", "val_windows",
                "parameter_count", "epochs_completed", "best_epoch", "best_val_auc"]:
        print(f"{key:36s}: {config[key]}")
    print("threshold:", threshold)

    epochs = [row["epoch"] for row in history]
    figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4))
    left.plot(epochs, [r["train_loss"] for r in history], label="train loss")
    left.plot(epochs, [r["val_loss"] for r in history], label="val loss")
    right.plot(epochs, [r["val_auc"] for r in history], label="val AUC theo frame")
    right.plot(epochs, [r["val_ap"] for r in history], label="val AP theo frame")
    right.plot(epochs, [r["val_video_auc_median"] for r in history], label="val AUC trong video (trung vị)")
    for axis in (left, right):
        axis.axvline(config["best_epoch"], color="green", linestyle="--"); axis.grid(alpha=0.3); axis.legend(); axis.set_xlabel("Epoch")
    plt.show()

## 10. Đánh giá trên tập test
- UBI-Fights: AUC theo frame trên **toàn bộ** tập test chuẩn (`test_videos.csv`).
- `ubi_short`, `ubi_crowd_short`: cùng model nhưng video bình thường chỉ tính 60 s đầu, giống tập test của lần 1–2 — dùng để so sánh với lần trước.
- Surveillance Camera Fight: test chéo theo clip.
- Nhóm `_crowd`: riêng các video trung bình ≥ 5 người/frame.
- `fight_video_frame_auc_median`: AUC trung vị trong từng video đánh nhau (model có tìm được đoạn đánh nhau trong video không).

Ngưỡng lấy từ validation, không chỉnh theo nhãn test.

In [ ]:
with step('10. Đánh giá'):
    EVAL_DIR = ARTIFACT_DIR / "evaluation"
    run("scripts/evaluate_fight.py", *dataset_args(DATASETS_TEST), "--pose-root", POSE_ROOT,
        "--model", ARTIFACT_DIR / "fight_stgcn.pt", "--threshold", ARTIFACT_DIR / "fight_threshold.json",
        "--output-dir", EVAL_DIR, "--stride", EVAL_STRIDE, "--short-normal-seconds", SHORT_TEST_SECONDS, "--device", DEVICE)

## 11. Xem metrics và so sánh với lần trước

In [ ]:
with step('11. Xem metrics'):
    from IPython.display import Image, display
    evaluation = json.loads((EVAL_DIR / "evaluation.json").read_text())
    print("Tập test đầy đủ (official_protocol):", evaluation["official_protocol"],
          "| video thiếu khung xương:", evaluation["missing_pose_caches"])
    for level in ("frame_level", "clip_level"):
        for name, metrics in evaluation[level].items():
            print(f"\n{name} — {level}")
            for key in ["roc_auc", "average_precision", "f1", "precision", "recall", "specificity", "accuracy", "balanced_accuracy"]:
                value = metrics[key]
                print(f"  {key:18s}: {value if value is None else round(value, 4)}")
            if "fight_video_frame_auc_median" in metrics:
                print(f"  AUC trong video   : {metrics['fight_video_frame_auc_median']} (trung vị {metrics['fight_videos']} video đánh nhau)")
            print(f"  confusion: TN={metrics['true_negative']} FP={metrics['false_positive']} FN={metrics['false_negative']} TP={metrics['true_positive']}")

    previous = DRIVE_ROOT / "artifacts" / PREVIOUS_RUN / "evaluation" / "evaluation.json" if PREVIOUS_RUN else None
    if previous and previous.exists():
        old = json.loads(previous.read_text())
        print(f"\nSo sánh AUC {PREVIOUS_RUN} → {RUN_NAME} trên cùng tập test (lần trước dùng tập rút gọn thì so với nhóm _short):")
        for level in ("frame_level", "clip_level"):
            for name, before_metrics in old.get(level, {}).items():
                same = f"{name}_short" if old.get("shortened_test_videos") and f"{name}_short" in evaluation[level] else name
                now = evaluation[level].get(same, {}).get("roc_auc")
                before = before_metrics.get("roc_auc")
                if before is not None and now is not None:
                    print(f"  {name:10s} {level:11s}: {before:.3f} → {now:.3f} ({now - before:+.3f})  [{RUN_NAME}: {same}]")
    for image in sorted(EVAL_DIR.glob("*.png")):
        print(image.name); display(Image(filename=str(image)))

## 12. Đóng gói model
`fight_stgcn.pt` và `fight_threshold.json` đi cùng nhau; giải nén vào thư mục `models/` của repository. Bản ZIP luôn được lưu ở `MyDrive/SENTINEL/artifacts/<RUN_NAME>/`; chỉ mở hộp thoại tải về khi `AUTO_DISCONNECT = False` (khi treo máy không có ai bấm).

In [ ]:
with step('12. Đóng gói model'):
    import shutil, zipfile
    from google.colab import files
    bundle = Path("/content/sentinel_fight_runtime.zip")
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
        for name in ["fight_stgcn.pt", "fight_threshold.json"]:
            archive.write(ARTIFACT_DIR / name, arcname=f"models/{name}")
        for name in ["config.json", "split.json", "training_history.json"]:
            archive.write(ARTIFACT_DIR / name, arcname=f"models/metadata/fight_{name}")
        archive.write(EVAL_DIR / "evaluation.json", arcname="models/metadata/fight_evaluation.json")
    shutil.copy2(bundle, ARTIFACT_DIR / bundle.name)
    print(ARTIFACT_DIR / bundle.name, f"{bundle.stat().st_size / 1024**2:.1f} MB")
    if not AUTO_DISCONNECT:
        files.download(str(bundle))

## 13. Kết thúc: ngắt runtime
Khi `AUTO_DISCONNECT = True`, cell này ghi "Hoàn tất" vào `MyDrive/SENTINEL/artifacts/<RUN_NAME>/run_log.txt`, đồng bộ Drive rồi ngắt runtime để không tốn compute unit khi treo máy. Nếu một mục phía trên bị lỗi, runtime cũng tự ngắt và lý do được ghi vào cùng file log. Lần sau mở lại notebook: đọc `run_log.txt` để biết đã chạy tới đâu.

In [ ]:
finish_and_disconnect(f"Hoàn tất. Kết quả: {ARTIFACT_DIR / 'evaluation' / 'evaluation.json'}")